# Study 9: condition-stratified, prevalence-as-coherence (no linking function)

Tests the actual hypothesis: hearing generics about one **region** of semantic space (e.g. physical
features) raises coherence there and spreads to nearby features (other physical) but not distant ones
(diet, personality) — with the GP **length_scale** controlling how far the bump spreads.

The earlier pooled model erased this by treating everyone as hearing all-generic training. Here we
**stratify by condition**: each condition heard generics only about its own region, so its coherence
field has a localized bump. The cross-condition contrast — the *same* test feature rated differently
depending on which region was trained — is what identifies the length_scale.

Per condition `c`, the trained (generic-endorsed) features are:
- `diet` / `personality` / `physical`: the 15 train features of that category,
- `heterogeneous`: the 15 train features flagged `in_heterogenous` (5 per category).

Predicted prevalence for a test feature = its coherence `sigmoid(y_test)` (no linking function);
ratings scored by `Beta(mean=pz1, conc=kappa)` with one global `kappa`. `mu_0`, `length_scale`, and
`kappa` are shared across conditions. `mu_0` is now the **baseline** (untrained) coherence, and the
bump sits in each condition's trained region.

In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import sys, csv, pickle as pkl
sys.path.insert(0, ".")

import numpy as np
import jax
import jax.numpy as jnp
import blackjax
import matplotlib.pyplot as plt
from scipy.special import logsumexp as scipy_logsumexp
from scipy.special import betainc
from scipy.optimize import minimize_scalar
from pyvbmc import VBMC

from model_jax import make_log_density_fn_joint

print(f"backend: {jax.default_backend()}")

backend: cpu


In [2]:
# ---- features: per-condition trained set + shared test set ----
with open('../features/set2_features_dataframe.pkl', 'rb') as f:
    df = pkl.load(f)
feat_idx  = df.set_index('feature')
train_df  = df[df.split == 'train']

CONDITIONS  = ['diet', 'personality', 'physical', 'heterogeneous']
CAT_OF_COND = {'diet':'diet_preferences', 'personality':'personality_behaviors', 'physical':'physical'}
CAT_SHORT   = {'diet_preferences':'diet', 'personality_behaviors':'personality', 'physical':'physical'}

# trained (generic-endorsed) features per condition
x_train_cond, u_train_cond = {}, {}
for c in CONDITIONS:
    sub = train_df[train_df.in_heterogenous] if c == 'heterogeneous' else train_df[train_df.category == CAT_OF_COND[c]]
    x_train_cond[c] = jnp.array(sub[['x_2d', 'y_2d']].values)
    u_train_cond[c] = jnp.zeros(len(sub), dtype=jnp.int32)   # all generic
    print(f"  {c:<14}: {len(sub)} trained features")

# shared test set, aligned to the CSV rating columns below
CSV_TO_FEATURE = {
    'diet_can_eat_spicy_1':'can eat spicy food','diet_breakfast_late_1':'eat breakfast very late',
    'diet_five_meals_day_1':'eat five meals a day','diet_like_juice_pulp_1':'like juice with pulp',
    'diet_pepper_on_all_1':'put pepper on all their foods','pers_cry_easily_1':'cry easily',
    'pers_collect_rocks_1':'like to collect rocks','pers_like_to_dance_1':'like to dance',
    'pers_like_highfive_1':'like to give high-fives','pers_read_books_1':'like to read books',
    'phys_can_roll_tongue_1':'can roll their tongue','phys_can_snap_toes_1':'can snap with their toes',
    'phys_can_wiggle_ears_1':'can wiggle their ears','phys_cold_hands_feet_1':'have cold hands and feet',
    'phys_snore_sleep_1':'snore when they sleep'}
TEST_CSV_COLS      = list(CSV_TO_FEATURE.keys())
test_feature_names = list(CSV_TO_FEATURE.values())
x_test    = jnp.array([feat_idx.loc[n, ['x_2d','y_2d']].values for n in test_feature_names])  # (J,2)
test_trait = [CAT_SHORT[feat_idx.loc[n, 'category']] for n in test_feature_names]             # trait per test feat
J = x_test.shape[0]
print(f"test features: {J}  (traits: {dict(zip(*np.unique(test_trait, return_counts=True)))})")

  diet          : 15 trained features
  personality   : 15 trained features
  physical      : 15 trained features
  heterogeneous : 15 trained features
test features: 15  (traits: {np.str_('diet'): np.int64(5), np.str_('personality'): np.int64(5), np.str_('physical'): np.int64(5)})


In [3]:
# ---- ratings stratified by condition ----
with open('../../data/study9.csv') as f:
    rows = list(csv.reader(f))
hdr = rows[0]; col_idx = [hdr.index(c) for c in TEST_CSV_COLS]; cond_i = hdr.index('condition')

R, cond = [], []
for r in rows[3:]:
    try:
        R.append([int(r[i]) / 100.0 for i in col_idx])
    except (ValueError, IndexError):
        continue
    cond.append(r[cond_i])
R = np.array(R); cond = np.array(cond)
responses_cond = {c: jnp.array(R[cond == c]) for c in CONDITIONS}
for c in CONDITIONS:
    print(f"  {c:<14}: {responses_cond[c].shape[0]} participants")

  diet          : 101 participants
  personality   : 101 participants
  physical      : 101 participants
  heterogeneous : 99 participants


## Prevalence-as-coherence likelihood (single global kappa)

In [ ]:
H = 1.0 / 200.0
def prevalence_loglik(responses, pz1, kappa):
    r = np.asarray(responses, dtype=np.float64)
    p = np.clip(np.asarray(pz1, dtype=np.float64), 1e-4, 1 - 1e-4)
    a = kappa * p; b = kappa * (1.0 - p)
    lo = np.clip(r - H, 0.0, 1.0); hi = np.clip(r + H, 0.0, 1.0)
    P = betainc(a[None, :], b[None, :], hi) - betainc(a[None, :], b[None, :], lo)
    return float(np.sum(np.log(np.maximum(P, 1e-300))))

def fit_kappa_multi(pz1bar_by_cond):
    """One global kappa maximizing the summed prevalence log-lik across conditions."""
    def negll(lk):
        k = float(np.exp(lk))
        return -sum(prevalence_loglik(responses_cond[c], pz1bar_by_cond[c], k) for c in CONDITIONS)
    res = minimize_scalar(negll, bounds=(np.log(0.5), np.log(500.0)), method='bounded')
    return float(np.exp(res.x))

In [ ]:
FIXED_PARAMS = {'output_scale': 1.5, 'beta': 3.0}
N_WARMUP  = 300
N_SAMPLES = 800        # per condition; 4 chains/eval, so kept lighter
STEP      = 5
MAX_EVALS = 120        # VBMC budget (4 MCMC chains/eval -> this is the main cost knob)

def condition_pz1_draws(x_train_c, u_train_c, params):
    """Thinned predicted-prevalence draws (S, J) for one condition: sigmoid(y_test)."""
    ldf = make_log_density_fn_joint(u_train_c, x_train_c, x_test, params)
    ip  = {'training_coherences': jnp.zeros(x_train_c.shape[0]), 'test_coherences': jnp.zeros(J)}
    k = jax.random.PRNGKey(0); k, wk = jax.random.split(k)
    w = blackjax.window_adaptation(blackjax.nuts, ldf)
    (st, npar), _ = w.run(wk, ip, num_steps=N_WARMUP)
    nuts = blackjax.nuts(ldf, **npar); step = jax.jit(lambda s, key: nuts.step(key, s))
    tc = []
    for key in jax.random.split(k, N_SAMPLES)[:-1]:
        st, _ = step(st, key)
        tc.append(np.array(st.position['test_coherences']))
    return np.array(jax.nn.sigmoid(np.array(tc)[::STEP]))   # (S, J)

In [ ]:
eval_count = [0]
eval_log   = []

def log_joint(phi):
    """Sum of per-condition prevalence-as-coherence log-liks + prior. phi = [log_ls, mu_0]."""
    log_ls, mu_0 = float(phi[0]), float(phi[1])
    length_scale = float(np.exp(log_ls))
    eval_count[0] += 1
    print(f"  eval {eval_count[0]:3d}: ls={length_scale:.3f}  mu_0={mu_0:.3f}", end="  ")

    log_prior = float(-0.5*((log_ls-np.log(0.5))/1.5)**2) + float(-0.5*mu_0**2)
    params = {**FIXED_PARAMS, 'length_scale': length_scale, 'mu_0': mu_0}

    pz1_draws = {c: condition_pz1_draws(x_train_cond[c], u_train_cond[c], params) for c in CONDITIONS}
    pz1_bar   = {c: pz1_draws[c].mean(axis=0) for c in CONDITIONS}
    kappa     = fit_kappa_multi(pz1_bar)            # one global nuisance scalar

    total_ll, per_cond = 0.0, []
    for c in CONDITIONS:
        S = pz1_draws[c].shape[0]
        ll_c = float(scipy_logsumexp([prevalence_loglik(responses_cond[c], pz1_draws[c][s], kappa)
                                      for s in range(S)]) - np.log(S))
        total_ll += ll_c; per_cond.append(ll_c)

    noise_std = float(np.std(per_cond))   # rough; cross-condition spread of log-liks
    log_joint_val = float(total_ll + log_prior)
    print(f"kappa={kappa:6.2f}  log_lik={total_ll:.1f}  log_joint={log_joint_val:.1f}")
    eval_log.append((np.array(phi, dtype=float), log_joint_val, noise_std))
    return log_joint_val, max(noise_std, 1.0)

In [ ]:
# pyVBMC varG-squeeze patch (numpy-compat fix)
import pyvbmc.vbmc.variational_optimization as _vopt
if not getattr(_vopt._gp_log_joint, "_varg_squeeze_patch", False):
    _orig = _vopt._gp_log_joint
    def _patched(*a, **k):
        out = list(_orig(*a, **k)); out[2] = None if out[2] is None else float(np.squeeze(out[2])); return tuple(out)
    _patched._varg_squeeze_patch = True; _vopt._gp_log_joint = _patched
    print("patched _gp_log_joint (varG squeeze)")

x0  = np.array([np.log(0.3), 0.0])
lb  = np.array([np.log(0.03), -4.0])
ub  = np.array([np.log(5.0),   4.0])    # ~6x the cloud diameter (0.78); above this = trait-blind null
plb = np.array([np.log(0.08), -2.0])
pub = np.array([np.log(1.0),   2.0])
print(f"Start ls={np.exp(x0[0]):.2f}, mu_0={x0[1]:.2f}  | bounds ls in [{np.exp(lb[0]):.2f},{np.exp(ub[0]):.2f}]")

vbmc = VBMC(log_joint, x0, lb, ub, plb, pub,
            options={'specify_target_noise': True, 'max_fun_evals': MAX_EVALS})
vbmc_result, vbmc_stats = vbmc.optimize()

## Posterior over (length_scale, mu_0)

`length_scale` now has a real job: it sets how far each condition's coherence bump spreads. The
hypothesis predicts an **intermediate** value (within-trait generalization, not cross-trait); a
large value would mean the bump spreads everywhere (trait-blind).

In [ ]:
phi_samples, _ = vbmc_result.sample(int(1e4))
ls_samples  = np.exp(phi_samples[:, 0]); mu0_samples = phi_samples[:, 1]
print(f"Posterior median: ls={np.median(ls_samples):.3f}  mu_0={np.median(mu0_samples):.3f}")
print(f"95% CI ls:  [{np.percentile(ls_samples,2.5):.3f}, {np.percentile(ls_samples,97.5):.3f}]")
print(f"95% CI mu_0: [{np.percentile(mu0_samples,2.5):.3f}, {np.percentile(mu0_samples,97.5):.3f}]")
print(f"convergence: {vbmc_stats['convergence_status']}  (evals {vbmc_stats['func_count']}, ELBO {vbmc_stats['elbo']:.1f})")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(ls_samples, bins=50, density=True, color='steelblue', alpha=0.7)
axes[0].axvline(np.median(ls_samples), color='tomato', lw=2, label=f'median={np.median(ls_samples):.2f}')
axes[0].set_xlabel('length_scale'); axes[0].set_title('Posterior: length_scale'); axes[0].legend()
axes[1].hist(mu0_samples, bins=50, density=True, color='seagreen', alpha=0.7)
axes[1].axvline(np.median(mu0_samples), color='tomato', lw=2, label=f'median={np.median(mu0_samples):.2f}')
axes[1].set_xlabel('mu_0 (baseline coherence)'); axes[1].set_title('Posterior: mu_0'); axes[1].legend()
axes[2].scatter(ls_samples[::10], mu0_samples[::10], alpha=0.1, s=2, color='purple')
axes[2].set_xlabel('length_scale'); axes[2].set_ylabel('mu_0'); axes[2].set_title('Joint posterior')
plt.tight_layout(); plt.show()

## The hypothesis test: does the trained region's coherence bump match the data?

At the posterior-mean (ls, mu_0), get each condition's predicted coherence per test feature and compare
to that condition's empirical mean. The prediction: in each condition, **matching-trait** test features
(same trait as the trained region) should be elevated relative to off-trait test features — and the
empirical means should show the same pattern.

In [ ]:
ls_m, mu0_m = float(ls_samples.mean()), float(mu0_samples.mean())
params_best = {**FIXED_PARAMS, 'length_scale': ls_m, 'mu_0': mu0_m}
print(f"Forward pass at ls={ls_m:.3f}, mu_0={mu0_m:.3f}")

pred_cond = {c: condition_pz1_draws(x_train_cond[c], u_train_cond[c], params_best).mean(0) for c in CONDITIONS}
emp_cond  = {c: np.array(responses_cond[c]).mean(0) for c in CONDITIONS}
traits = np.array(test_trait)
tcolor = {'diet':'tab:green', 'personality':'tab:purple', 'physical':'tab:orange'}

# (1) predicted vs empirical per condition, colored by test-feature trait; matched trait outlined
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharex=True, sharey=True)
for ax, c in zip(axes, CONDITIONS):
    for tr in tcolor:
        m = traits == tr
        edge = 'k' if (tr == c) else 'none'          # matched-trait points outlined
        ax.scatter(emp_cond[c][m], pred_cond[c][m], s=70, color=tcolor[tr],
                   edgecolor=edge, linewidth=1.5, label=tr)
    ax.plot([0,1],[0,1],'--',color='gray',alpha=0.5); ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_title(f"{c}  (matched trait outlined)"); ax.set_xlabel('empirical mean')
axes[0].set_ylabel('predicted coherence'); axes[0].legend(fontsize=8, title='test trait')
plt.tight_layout(); plt.show()

# (2) condition x test-trait elevation: predicted vs empirical (diagonal = matched -> should pop)
def by_trait(d):  # mean over test features within each trait, per condition
    return np.array([[d[c][traits==tr].mean() for tr in ['diet','personality','physical']] for c in CONDITIONS])
pred_mat, emp_mat = by_trait(pred_cond), by_trait(emp_cond)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, mat, ttl in [(axes[0], pred_mat, 'PREDICTED coherence'), (axes[1], emp_mat, 'EMPIRICAL mean rating')]:
    im = ax.imshow(mat, aspect='auto', cmap='viridis')
    ax.set_xticks(range(3)); ax.set_xticklabels(['diet','personality','physical'])
    ax.set_yticks(range(4)); ax.set_yticklabels(CONDITIONS)
    ax.set_xlabel('test-feature trait'); ax.set_ylabel('condition (trained region)'); ax.set_title(ttl)
    for i in range(4):
        for j in range(3):
            ax.text(j, i, f"{mat[i,j]:.2f}", ha='center', va='center', fontsize=9,
                    color='white' if mat[i,j] < mat.mean() else 'black')
    plt.colorbar(im, ax=ax)
fig.suptitle('Does training a region raise matched-trait test features? (rows: trained; cols: tested)')
plt.tight_layout(); plt.show()